# Set Library Path

In [5]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [6]:
library(enrichR)
library(dplyr)
library(openxlsx)
library(ggplot2)
library(readxl)
library(tidyr)

# Load Data

In [7]:
# List all DEG files in the subdirectories 
files <- list.files(path = "05_DEGs", pattern = "_results.csv$", full.names = TRUE, recursive = TRUE)

# Initialize an empty vector to hold files with significant DEGs
valid_files <- c()

# Initialize a data frame to store summary statistics
summary_table <- data.frame(
  Comparison = character(),
  DEGs_adj.P.Val_0.01 = integer(),
  logFC_0.5 = integer(),
  logFC_1 = integer(),
  logFC_1.5 = integer(),
  logFC_2 = integer(),
  logFC_2.5 = integer(),
  logFC_3 = integer(),
  logFC_3plus = integer(),
  stringsAsFactors = FALSE
)

# Loop through each file
for (file in files) {
  
  # Read the current file
  df <- read.csv(file, row.names = 1)
  
  # Extract comparison name 
  comparison <- gsub("_results.csv", "", basename(file))
  
  # Filter to only significant DEGs (adj.P.Val < 0.05)
  significant_df <- df[df$adj.P.Val < 0.05 & !is.na(df$adj.P.Val), ]
  
  # Check the number of significant DEGs
  num_significant <- nrow(significant_df)
  
  print(paste("File:", basename(file), "- Number of significant DEGs:", num_significant))
  
  # If there are significant DEGs, keep the file
  if (num_significant > 0) {
    valid_files <- c(valid_files, file)
    
    # Calculate DEG counts by logFC thresholds
    logfc_abs <- abs(significant_df$logFC)
    
    logFC_0.5 <- sum(logfc_abs >= 0.5, na.rm = TRUE)
    logFC_1 <- sum(logfc_abs >= 1, na.rm = TRUE)
    logFC_1.5 <- sum(logfc_abs >= 1.5, na.rm = TRUE)
    logFC_2 <- sum(logfc_abs >= 2, na.rm = TRUE)
    logFC_2.5 <- sum(logfc_abs >= 2.5, na.rm = TRUE)
    logFC_3 <- sum(logfc_abs >= 3, na.rm = TRUE)
    logFC_3plus <- sum(logfc_abs > 3, na.rm = TRUE)
    
    # Add to summary table
    summary_table <- rbind(summary_table, data.frame(
      Comparison = comparison,
      DEGs_adj.P.Val_0.05 = num_significant,
      logFC_0.5 = logFC_0.5,
      logFC_1 = logFC_1,
      logFC_1.5 = logFC_1.5,
      logFC_2 = logFC_2,
      logFC_2.5 = logFC_2.5,
      logFC_3 = logFC_3,
      logFC_3plus = logFC_3plus,
      stringsAsFactors = FALSE
    ))
  }
}

# Update the original files list to only include valid files
files <- valid_files

# View
cat("\n=== Summary Table of DEGs by logFC Thresholds ===\n")
print(summary_table)

# Save
write.csv(summary_table, "00_TABLES/deg_summary_table.csv", row.names = FALSE)

[1] "File: R270X_D18_vs_R270X_D46_results.csv - Number of significant DEGs: 9275"
[1] "File: R270X_D18_vs_R270X_D90_results.csv - Number of significant DEGs: 11725"
[1] "File: R270X_D18_vs_WT_D18_results.csv - Number of significant DEGs: 130"
[1] "File: R270X_D18_vs_WT_D46_results.csv - Number of significant DEGs: 9315"
[1] "File: R270X_D18_vs_WT_D90_results.csv - Number of significant DEGs: 11562"
[1] "File: R270X_D46_vs_R270X_D90_results.csv - Number of significant DEGs: 4900"
[1] "File: R270X_D46_vs_WT_D46_results.csv - Number of significant DEGs: 91"
[1] "File: R270X_D46_vs_WT_D90_results.csv - Number of significant DEGs: 4960"
[1] "File: R270X_D90_vs_WT_D90_results.csv - Number of significant DEGs: 66"
[1] "File: WT_D18_vs_R270X_D46_results.csv - Number of significant DEGs: 9422"
[1] "File: WT_D18_vs_R270X_D90_results.csv - Number of significant DEGs: 11704"
[1] "File: WT_D18_vs_WT_D46_results.csv - Number of significant DEGs: 9438"
[1] "File: WT_D18_vs_WT_D90_results.csv - Number

# Plot DEGs

In [8]:
# Prepare data for plotting
plot_data <- summary_table %>%
  # Select only the logFC columns (excluding total DEGs column)
  mutate(Comparison = gsub("_", " ", Comparison)) %>%  
  select(Comparison, starts_with("logFC_")) %>%
  # Reshape from wide to long format for stacking
  pivot_longer(cols = starts_with("logFC_"),
               names_to = "logFC_threshold",
               values_to = "count") %>%
  # Clean up threshold labels for better readability
  mutate(logFC_threshold = case_when(
    logFC_threshold == "logFC_0.5" ~ "|logFC| ≥ 0.5",
    logFC_threshold == "logFC_1" ~ "|logFC| ≥ 1",
    logFC_threshold == "logFC_1.5" ~ "|logFC| ≥ 1.5",
    logFC_threshold == "logFC_2" ~ "|logFC| ≥ 2",
    logFC_threshold == "logFC_2.5" ~ "|logFC| ≥ 2.5",
    logFC_threshold == "logFC_3" ~ "|logFC| ≥ 3",
    logFC_threshold == "logFC_3plus" ~ "|logFC| > 3",
    TRUE ~ logFC_threshold
  )) %>%
  # Convert to factor to control stacking order (largest threshold on bottom)
  mutate(logFC_threshold = factor(logFC_threshold,
                                   levels = c("|logFC| > 3", "|logFC| ≥ 3", 
                                              "|logFC| ≥ 2.5", "|logFC| ≥ 2",
                                              "|logFC| ≥ 1.5", "|logFC| ≥ 1",
                                              "|logFC| ≥ 0.5")))

In [16]:
# Create stacked bar plot
stacked_plot <- ggplot(plot_data, aes(x = Comparison, y = count, fill = logFC_threshold)) +
  geom_bar(stat = "identity", position = "stack") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
        axis.title = element_text(size = 12),
        legend.title = element_text(size = 10),
        legend.text = element_text(size = 8),
        legend.position = "right") +
  labs(x = "DEG Comparison",
       y = "Number of Significant DEGs",
       title = "Differential Expression Summary by logFC Threshold",
       fill = "logFC Cutoff") +
  scale_fill_brewer(palette = "Blues", direction = -1) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05)))

# Save plot
ggsave("07_DEG_Figures/Bar_Plots/deg_stacked_barplot.pdf", stacked_plot, width = 12, height = 8)
ggsave("07_DEG_Figures/Bar_Plots/deg_stacked_barplot.png", stacked_plot, width = 12, height = 8, dpi = 600)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 3' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 2.5' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 2' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 1.5' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 1' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 0.5' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”


In [11]:
# Create a horizontal version 
horizontal_plot <- ggplot(plot_data, aes(x = reorder(Comparison, count, sum), y = count, fill = logFC_threshold)) +
  geom_bar(stat = "identity", position = "stack") +
  coord_flip() +
  theme_classic() +
  theme(axis.text.y = element_text(size = 6),
        axis.title = element_text(size = 12),
        legend.title = element_text(size = 10),
        legend.text = element_text(size = 8),
        legend.position = "right") +
  labs(x = "DEG Comparison",
       y = "Number of Significant DEGs",
       title = "Differential Expression Summary by logFC Threshold",
       fill = "logFC Cutoff") +
  scale_fill_brewer(palette = "Blues", direction = -1) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05)))

# Save horizontal version
ggsave("07_DEG_Figures/Bar_Plots/deg_stacked_barplot_horizontal.pdf", horizontal_plot, width = 7, height = 4)
ggsave("07_DEG_Figures/Bar_Plots/deg_stacked_barplot_horizontal.png", horizontal_plot, width = 7, height = 4, dpi = 600)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 3' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 2.5' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 2' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 1.5' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 1' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '|logFC| ≥ 0.5' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”


In [12]:
# Each DEG counted only in its highest threshold
plot_data_non_cumulative <- summary_table %>%
  mutate(Comparison = gsub("_", " ", Comparison)) %>%
  mutate(
    logFC_3plus_only = logFC_3,                                       # |logFC| ≥ 3
    logFC_2.5_only = logFC_2.5 - logFC_3,                             # 2.5 ≤ |logFC| < 3
    logFC_2_only = logFC_2 - logFC_2.5,                               # 2 ≤ |logFC| < 2.5
    logFC_1.5_only = logFC_1.5 - logFC_2,                             # 1.5 ≤ |logFC| < 2
    logFC_1_only = logFC_1 - logFC_1.5,                               # 1 ≤ |logFC| < 1.5
    logFC_0.5_only = logFC_0.5 - logFC_1                              # 0.5 ≤ |logFC| < 1
  ) %>%
  select(Comparison, contains("_only")) %>%
  pivot_longer(cols = contains("_only"),
               names_to = "logFC_threshold",
               values_to = "count") %>%
  mutate(logFC_threshold = case_when(
    logFC_threshold == "logFC_0.5_only" ~ "0.5-1",
    logFC_threshold == "logFC_1_only" ~ "1-1.5",
    logFC_threshold == "logFC_1.5_only" ~ "1.5-2",
    logFC_threshold == "logFC_2_only" ~ "2-2.5",
    logFC_threshold == "logFC_2.5_only" ~ "2.5-3",
    logFC_threshold == "logFC_3plus_only" ~ "≥3",
    TRUE ~ logFC_threshold
  ))

# Define legend order (using regular hyphens to match the labels)
plot_data_non_cumulative <- plot_data_non_cumulative %>%
  mutate(logFC_threshold = factor(logFC_threshold,
                                   levels = c("0.5-1", "1-1.5", "1.5-2", 
                                              "2-2.5", "2.5-3", "≥3")))

In [18]:
# Create non-cumulative stacked bar plot
non_cumulative_plot <- ggplot(plot_data_non_cumulative, aes(x = Comparison, y = count, fill = logFC_threshold)) +
  geom_bar(stat = "identity", position = "stack") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 6),
        axis.title = element_text(size = 12),
        legend.title = element_text(size = 10),
        legend.text = element_text(size = 8)) +
  labs(x = "DEG Comparison",
       y = "Number of Significant DEGs",
       title = "Differential Expression Summary by logFC Threshold (Non-cumulative)",
       fill = "|logFC| Range") +
  scale_fill_brewer(palette = "Blues") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05)))

ggsave("07_DEG_Figures/Bar_Plots/deg_stacked_barplot_non_cumulative.pdf", non_cumulative_plot, width = 10, height = 7)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '≥3' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”


In [14]:
# Create horizontal non-cumulative stacked bar plot
non_cumulative_plot_horizontal <- ggplot(plot_data_non_cumulative, aes(x = reorder(Comparison, count, sum), y = count, fill = logFC_threshold)) +
  geom_bar(stat = "identity", position = "stack") +
  coord_flip() +
  theme_classic() +
  theme(axis.text.y = element_text(size = 6),
        axis.title = element_text(size = 12),
        legend.title = element_text(size = 10),
        legend.text = element_text(size = 8),
        legend.position = "right") +
  labs(x = "DEG Comparison",
       y = "Number of Significant DEGs",
       title = "Differential Expression Summary by logFC Threshold\n(Non-cumulative)",
       fill = "|logFC| Range") +
  scale_fill_brewer(palette = "Blues") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05)))

# Save horizontal plot
ggsave("07_DEG_Figures/Bar_Plots/deg_stacked_barplot_non_cumulative_horizontal.pdf", non_cumulative_plot_horizontal, width = 7, height = 4)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '≥3' in 'mbcsToSbcs': >= substituted for ≥ (U+2265)”
